In [1]:
from torch.optim import AdamW, Optimizer
import torch.nn.functional as F

In [2]:
from config import SAVE_DATA_PATH
from data.load_data import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(SAVE_DATA_PATH)

Loading dataset from disk:   0%|          | 0/18 [00:00<?, ?it/s]

In [3]:
from model.load_model import get_unires_model


model = get_unires_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.l

In [4]:
from config import LEARNING_RATE


optimizer = AdamW(model.parameters(), LEARNING_RATE)

In [24]:
import torch

from config import DICE_SMOOTH


def get_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # (B,224,224)
    probs = torch.sigmoid(logits)

    # flatten
    batch_size = logits.shape[0]
    # (B,50176)
    probs = probs.view(batch_size, -1)
    # (B,50176)
    targets = targets.view(batch_size, -1)

    # (B,)
    numerator = 2 * (probs * targets).sum(dim=1)
    # (B,)
    denominator = probs.sum(dim=1) + targets.sum(dim=1)
    # (B,)
    score = (numerator + DICE_SMOOTH) / (denominator + DICE_SMOOTH)
    loss = (1 - score).mean()

    return loss


def loss_fn(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    bce_loss = F.binary_cross_entropy_with_logits(logits, targets)
    dice_loss = get_dice_loss(logits, targets)
    return bce_loss + dice_loss

In [ ]:
from config import SEED


torch.manual_seed(SEED)
loss_fn(torch.rand(16, 224, 224).to("cuda"), torch.rand(16, 224, 224).to("cuda"))

tensor(1.1806, device='cuda:0')

In [ ]:
loss_fn(
    torch.rand(16, 224, 224).to("cuda"),
    torch.randint(0, 2, (16, 224, 224)).float().to("cuda"),
)

tensor(1.1800, device='cuda:0')

In [8]:
type(optimizer)

torch.optim.adamw.AdamW

In [9]:
isinstance(optimizer, Optimizer)

True

In [10]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()

In [11]:
isinstance(writer, SummaryWriter)

True

In [12]:
from typing import Tuple

from config import IOU_EPSILON


def calculate_miou_oiou(
    preds: torch.Tensor, targets: torch.Tensor
) -> Tuple[float, float]:
    # (B,224,224)
    preds = preds.bool()
    # (B,224,224)
    targets = targets.bool()

    # (B,)
    intersection = (preds & targets).sum(dim=(1, 2)).float()
    # (B,)
    union = (preds | targets).sum(dim=(1, 2)).float()

    # (B,)
    sample_ious = intersection / (union + IOU_EPSILON)
    miou = sample_ious.mean().item()

    total_intersection = intersection.sum().item()
    total_union = union.sum().item()
    oiou = total_intersection / (total_union + IOU_EPSILON)

    return miou, oiou

In [13]:
type((1, 2))

tuple

In [14]:
def log_metrics(
    writer: SummaryWriter,
    log_tag: str,  # train/validation/test
    global_step: int,
    preds: torch.Tensor,
    targets: torch.Tensor,
    loss: float,
) -> None:
    accuracy = (targets == preds).float().mean().item()
    miou, oiou = calculate_miou_oiou(preds, targets)

    writer.add_scalars(
        log_tag,
        {
            "loss": loss,
            "accuracy": accuracy,
            "miou": miou,
            "oiou": oiou,
        },
        global_step,
    )

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [16]:
# from config import BATCH_SIZE, LOG_INTERVAL
# from model.unires import UniRes


# def train_step(model: UniRes, batch) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
#     model.train()

#     pixel_values = batch["pixel_values"].to(device)
#     input_ids = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)
#     targets = batch["seg_masks"].to(device)

#     logits = model(pixel_values, input_ids, attention_mask)
#     loss = total_loss(logits, targets.float())

#     return logits, targets, loss

In [17]:
# log_metrics(
#     "train",
#     0,
#     torch.randint(0, 2, (16, 224, 224)),
#     torch.randint(0, 2, (16, 224, 224)),
#     0.3,
# )

In [18]:
len(train_loader)

440

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model.to(device)

cuda


UniRes(
  (image_encoder): UniResImageEncoder(
    (image_encoder): CLIPVisionModel(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
        (position_embedding): Embedding(50, 768)
      )
      (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1):

In [20]:
type(train_loader)

torch.utils.data.dataloader.DataLoader

In [28]:
from typing import Callable

from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Optimizer

from config import LOG_INTERVAL
from model.unires import UniRes


def train_loop(
    dataloader: DataLoader,
    model: UniRes,
    optimizer: Optimizer,
    loss_fn: Callable,
    writer: SummaryWriter,
    device: str,
    epoch_idx: int,
) -> None:
    model.train()
    num_batches = len(dataloader)

    for batch_idx, batch in enumerate(tqdm(dataloader)):
        # extract batch data
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["seg_masks"].to(device)

        # forward & loss
        logits = model(pixel_values, input_ids, attention_mask)
        loss = loss_fn(logits, targets.float())

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # logging
        if batch_idx % LOG_INTERVAL == 0:
            preds = logits > 0
            global_step = num_batches * epoch_idx + batch_idx
            accuracy = (targets == preds).float().mean().item()
            miou, oiou = calculate_miou_oiou(preds, targets)

            writer.add_scalars(
                "train",
                {
                    "loss": loss,
                    "accuracy": accuracy,
                    "miou": miou,
                    "oiou": oiou,
                },
                global_step,
            )

In [29]:
def eval_loop(
    dataloader: DataLoader,
    model: UniRes,
    loss_fn: Callable,
    writer: SummaryWriter,
    device: str,
    epoch_idx: int,
) -> None:
    model.eval()

    num_batches = len(dataloader)
    total_loss = 0
    total_accuracy = 0
    total_miou = 0
    total_oiou = 0

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader)):
            # extract batch data
            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["seg_masks"].to(device)

            # forward & loss
            logits = model(pixel_values, input_ids, attention_mask)
            loss = loss_fn(logits, targets.float())

            # logging
            preds = logits > 0
            accuracy = (targets == preds).float().mean().item()
            miou, oiou = calculate_miou_oiou(preds, targets)

            total_loss += loss.item()
            total_accuracy += accuracy
            total_miou += miou
            total_oiou += oiou

    # write log
    writer.add_scalars(
        "validate",
        {
            "loss": total_loss / num_batches,
            "accuracy": total_accuracy / num_batches,
            "miou": total_miou / num_batches,
            "oiou": total_oiou / num_batches,
        },
        epoch_idx,
    )

In [25]:
len(val_loader)

55

In [26]:
len(train_loader.dataset)

14076

In [32]:
from config import MAX_EPOCHS


writer = SummaryWriter()

for epoch in range(1):
    # train_loop(train_loader, model, optimizer, loss_fn, writer, device, epoch)
    eval_loop(val_loader, model, loss_fn, writer, device, epoch)

100%|██████████| 55/55 [01:21<00:00,  1.48s/it]
